# DNB — Offline Smoke Tests

Phase-prediction scheduling:

```
Detect at rising zero crossing (3π/2) → predict peak (0) → schedule stim
```

Phase map: `0=peak  π/2=falling  π=trough  3π/2=rising  2π=peak`

The detector fires at `detection_phase` (rising zero crossing).
The trigger uses the detected frequency to predict when `stim_phase`
(peak) will occur — a quarter period later — and schedules stims.

**Event semantics:**
- `SLOW_WAVE` — detection at rising zero crossing. Carries `delay_to_stim_ms`.
- `STIM` — scheduled stimulation at predicted peak. `pulse_index` 1-indexed.

In [ ]:
from math import pi

CFG = {
    'pipeline': {
        'sample_rate': 1000.0,
        'n_channels': 1,
        'buffer_duration': 10.0,
        'chunk_duration': 0.1,
    },
    'wavelet': {
        'freq_min': 0.5,
        'freq_max': 4.0,
        'n_freqs': 10,
        'n_cycles_base': 3.0,
    },
    'target_wave': {
        'id': 'slow_wave',
        'freq_range': (0.5, 4.0),
        'detection_phase': pi,  # detect at RISING ZERO CROSSING
        'phase_tolerance': 0.05,
        'amp_min': 1000.0,
        'amp_max': 10000.0,
        'warmup_chunks': 5,
        'amp_smoothing': 5,
    },
    'trigger': {
        'activation_detector_id': 'slow_wave',
        'inhibition_detector_id': 'ied_monitor',
        'n_pulses': 1,
        'stim_phase': 0.0,           # stim at PEAK
        'backoff_s': 2.5,
        'inhibition_cooldown_s': 2.5,
    },
    'amplitude_monitor': {
        'enabled': True,
        'id': 'ied_monitor',
        'freq_range': (80.0, 120.0),
        'threshold': None,
        'adaptive_n_std': 5.0,
        'warmup_chunks': 5,
        'baseline_chunks': 100,
    },
    'synthetic': {
        'duration_s': 120.0,
        'snr': 5.0,
        'n_slow_waves': 15,
        'n_ieds': 40,
        'sw_amplitude': 500.0,
        'ied_amplitude': 3000.0,
        'seed': 42,
    },
    'validation': {
        'time_tolerance': 0.5,
    },
}

print('Config loaded.')
print(f"  detection_phase = {CFG['target_wave']['detection_phase']:.2f} rad ({CFG['target_wave']['detection_phase']*180/pi:.0f}°)")
print(f"  stim_phase      = {CFG['trigger']['stim_phase']:.2f} rad ({CFG['trigger']['stim_phase']*180/pi:.0f}°)")
print(f"  n_pulses        = {CFG['trigger']['n_pulses']}")
print(f"  chunk_duration  = {CFG['pipeline']['chunk_duration']}s")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import (
    WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger,
)
from dnb.validation.ground_truth import validate, Annotation
from test_data import TestData

import dnb
print(f'DNB v{dnb.__version__}')

td = TestData()

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
def make_pipeline_config():
    p = CFG['pipeline']
    return PipelineConfig(
        sample_rate=p['sample_rate'], n_channels=p['n_channels'],
        buffer_duration=p['buffer_duration'], chunk_duration=p['chunk_duration'],
    )

def make_wavelet():
    w = CFG['wavelet']
    return WaveletConvolution(
        freq_min=w['freq_min'], freq_max=w['freq_max'],
        n_freqs=w['n_freqs'], n_cycles_base=w['n_cycles_base'],
    )

def make_detector(**overrides):
    tw = {**CFG['target_wave'], **overrides}
    return TargetWaveDetector(
        id=tw['id'], freq_range=tw['freq_range'],
        detection_phase=tw['detection_phase'],
        phase_tolerance=tw['phase_tolerance'],
        amp_min=tw['amp_min'], amp_max=tw['amp_max'],
        warmup_chunks=tw['warmup_chunks'], amp_smoothing=tw['amp_smoothing'],
    )

def make_inhibitor(**overrides):
    am = {**CFG['amplitude_monitor'], **overrides}
    return AmplitudeMonitor(
        id=am['id'], freq_range=am['freq_range'],
        threshold=am['threshold'], adaptive_n_std=am['adaptive_n_std'],
        warmup_chunks=am['warmup_chunks'], baseline_chunks=am['baseline_chunks'],
    )

def make_trigger(**overrides):
    tr = {**CFG['trigger'], **overrides}
    return StimTrigger(
        activation_detector_id=tr['activation_detector_id'],
        inhibition_detector_id=tr['inhibition_detector_id'],
        n_pulses=tr['n_pulses'],
        stim_phase=tr['stim_phase'],
        backoff_s=tr['backoff_s'],
        inhibition_cooldown_s=tr['inhibition_cooldown_s'],
    )

def get_detections(events):
    return [e for e in events if e.event_type == EventType.SLOW_WAVE]

def get_stims(events, pulse_index=None):
    out = [e for e in events if e.event_type == EventType.STIM]
    if pulse_index is not None:
        out = [e for e in out if e.metadata.get('pulse_index') == pulse_index]
    return out

print('Helpers ready.')

---
## 1. Clean sine — rising zero crossing detection → peak stim

With a 1 Hz sine, detection at 3π/2 (rising zero crossing) should schedule stim
250ms later at 0 (positive peak).

In [ ]:
path = td.clean_sine()

pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        make_wavelet(),
        make_detector(amp_min=100.0, amp_max=100000.0, phase_tolerance=0.1),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=0.0),
    ],
    config=make_pipeline_config(),
)

events = pipeline.run_offline()
detections = get_detections(events)
stims = get_stims(events)

print(f'{len(detections)} detections, {len(stims)} stims')

if detections and stims:
    det_times = np.array([e.timestamp for e in detections])
    stim_times = np.array([e.timestamp for e in stims])
    det_phases = np.array([e.metadata['detection_phase'] for e in detections])
    delays = np.array([e.metadata.get('delay_to_stim_ms', 0) for e in detections])

    print(f'Detection phase: mean={np.mean(det_phases):.3f} rad '
          f'(target={CFG["target_wave"]["detection_phase"]:.3f})')
    print(f'Predicted delay to stim: {np.mean(delays):.0f} ± {np.std(delays):.0f} ms')

    # Verify stims land at actual peaks
    data = np.load(str(path))
    sig = data['continuous'][0]
    sr = CFG['pipeline']['sample_rate']
    t = np.arange(len(sig)) / sr

    # Check signal value at stim times — should be near +amplitude
    stim_values = [sig[min(int(s * sr), len(sig)-1)] for s in stim_times[:20]]
    print(f'Signal at stim times: mean={np.mean(stim_values):.0f} '
          f'(should be near +500, the peak)')

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(t, sig, 'b-', lw=0.5, alpha=0.7)
    for e in detections[:30]:
        axes[0].axvline(e.timestamp, color='blue', alpha=0.3, lw=1)
    for e in stims[:30]:
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1)
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].set_title('1 Hz sine — blue=detection (rising zero), red=stim (predicted peak)')

    # Phase at detection times
    axes[1].scatter(det_times[:30], det_phases[:30], c='blue', s=20, label='detection phase')
    axes[1].axhline(3*pi/2, color='blue', ls='--', label=f'target=3π/2 (rising zero)')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

---
## 2. Synthetic slow waves — detection + stim validation

In [ ]:
syn = CFG['synthetic']
path_sw, gt_events = td.slow_waves(snr=syn['snr'])
sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Planted: {len(sw_gt)} slow waves')

data = np.load(str(path_sw))
sig = data['continuous'][0]
t = np.arange(len(sig)) / CFG['pipeline']['sample_rate']

pipeline = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=2.5),
    ],
    config=make_pipeline_config(),
)
all_events = pipeline.run_offline()
detections = get_detections(all_events)
stims = get_stims(all_events)
print(f'Detections (at trough): {len(detections)}')
print(f'Stims (at predicted peak): {len(stims)}')

# Validate: stims should land near the GT slow wave peaks
# GT timestamps mark the centre of planted SWs — the peak
annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(stims, annotations, time_tolerance=CFG['validation']['time_tolerance'])
print(report.summary())

# Print delay stats
if detections:
    delays = [e.metadata.get('delay_to_stim_ms', 0) for e in detections]
    print(f'Detection→Stim delay: {np.mean(delays):.0f} ± {np.std(delays):.0f} ms')

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2,
               label='GT peak' if e == sw_gt[0] else '')
for i, e in enumerate(detections):
    ax.axvline(e.timestamp, color='blue', alpha=0.3, lw=1,
               label='detection (trough)' if i == 0 else '')
for i, e in enumerate(stims):
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--',
               label='stim (predicted peak)' if i == 0 else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Green=GT peak, Blue=detection at rising zero, Red=stim at predicted peak')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. N-pulse scheduling

- `n_pulses=0` → SLOW_WAVE only
- `n_pulses=1` → 1 stim at first predicted peak
- `n_pulses=3` → stim at first peak + 2 more at next predicted peaks

In [ ]:
for n_pulses in [0, 1, 3]:
    pipe = Pipeline(
        source=FileSource(str(path_sw)),
        modules=[
            make_wavelet(),
            make_detector(),
            make_trigger(n_pulses=n_pulses, inhibition_detector_id=None, backoff_s=2.5),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    all_stims = get_stims(dets)
    print(f'n_pulses={n_pulses}: {len(sw_dets)} detections, {len(all_stims)} stims')

    if n_pulses >= 1 and all_stims:
        for d in sw_dets[:2]:
            det_t = d.timestamp
            freq = d.metadata['frequency']
            delay_ms = d.metadata.get('delay_to_stim_ms', 0)
            my_stims = sorted(
                [s for s in all_stims if abs(s.metadata.get('detection_time', -1) - det_t) < 0.001],
                key=lambda e: e.metadata['pulse_index'],
            )
            print(f'  det t={det_t:.3f}s freq={freq:.2f}Hz delay_to_stim={delay_ms:.0f}ms:')
            for s in my_stims:
                k = s.metadata['pulse_index']
                delay_from_det = (s.timestamp - det_t) * 1000
                print(f'    stim {k}/{n_pulses} t={s.timestamp:.3f}s '
                      f'(+{delay_from_det:.0f}ms from detection)')
    print()

In [ ]:
pipe_3 = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=3, inhibition_detector_id=None, backoff_s=5.0),
    ],
    config=make_pipeline_config(),
)
dets_3 = pipe_3.run_offline()
det_events_3 = get_detections(dets_3)
stims_3 = get_stims(dets_3)

pulse_colors = {1: 'red', 2: 'orange', 3: 'purple'}
labels_used = set()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2,
               label='GT' if e == sw_gt[0] else '')
for i, e in enumerate(det_events_3):
    ax.axvline(e.timestamp, color='blue', alpha=0.2, lw=1,
               label='detection' if i == 0 else '')
for e in stims_3:
    pi_idx = e.metadata['pulse_index']
    c = pulse_colors.get(pi_idx, 'gray')
    lbl = f'stim {pi_idx}' if pi_idx not in labels_used else ''
    labels_used.add(pi_idx)
    ax.axvline(e.timestamp, color=c, alpha=0.7, lw=1.2, ls='--', label=lbl)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('3-pulse: detect at trough → stim at predicted peaks')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 3b. Verify stim timing

Stim k should land at `det_time + Δt + (k-1)/freq` where Δt is the
phase delay from detection (trough) to stim (peak).

In [ ]:
from collections import defaultdict

sequences = defaultdict(list)
for e in stims_3:
    sequences[e.metadata['detection_time']].append(e)

all_errors = []
for det_t, seq in sorted(sequences.items()):
    seq.sort(key=lambda e: e.metadata['pulse_index'])
    freq = seq[0].metadata['frequency']
    # Find the corresponding detection event
    det_event = next((d for d in det_events_3 if abs(d.timestamp - det_t) < 0.001), None)
    if det_event is None:
        continue
    delay_s = det_event.metadata.get('delay_to_stim_ms', 500) / 1000
    period = 1.0 / freq
    for e in seq:
        k = e.metadata['pulse_index']
        expected = det_t + delay_s + (k - 1) * period
        all_errors.append(e.timestamp - expected)

errs = np.array(all_errors) * 1000
print(f'{len(sequences)} sequences, {len(all_errors)} stim events')
print(f'Timing error: {np.mean(errs):.1f} ± {np.std(errs):.1f} ms')
print(f'  (quantised to chunk_duration={CFG["pipeline"]["chunk_duration"]*1000:.0f} ms)')

---
## 4. IED inhibition

In [ ]:
path_ied, gt_ied = td.slow_waves_with_ieds(
    snr=syn['snr'], ied_amplitude=syn['ied_amplitude'],
)
sw_gt_ied = [e for e in gt_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

data_ied = np.load(str(path_ied))
sig_ied = data_ied['continuous'][0]
t_ied = np.arange(len(sig_ied)) / CFG['pipeline']['sample_rate']

# WITH inhibition
pipe_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(), make_inhibitor(),
             make_trigger(n_pulses=1, backoff_s=2.5)],
    config=make_pipeline_config(),
)
stims_inh = get_stims(pipe_inh.run_offline())

# WITHOUT inhibition
pipe_no = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(),
             make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=3.0)],
    config=make_pipeline_config(),
)
stims_no = get_stims(pipe_no.run_offline())

def near_ieds(stims, ieds, win=1.0):
    return sum(1 for s in stims if any(abs(s.timestamp - i.timestamp) < win for i in ieds))

print(f'With inhibition:    {len(stims_inh)} stims, {near_ieds(stims_inh, ied_gt)} near IEDs')
print(f'Without inhibition: {len(stims_no)} stims, {near_ieds(stims_no, ied_gt)} near IEDs')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, stims, label in [
    (axes[0], stims_no, 'WITHOUT inhibition'),
    (axes[1], stims_inh, 'WITH inhibition'),
]:
    ax.plot(t_ied, sig_ied, 'b-', lw=0.3, alpha=0.5)
    for e in sw_gt_ied:
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2)
    for e in ied_gt:
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3)
    for e in stims:
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--')
    ax.set_ylabel('Amp')
    ax.set_title(label)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

### 4b. N-pulse with IED inhibition

Inhibition should cancel pending scheduled stims.

In [ ]:
pipe_3_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(), make_inhibitor(),
             make_trigger(n_pulses=3, backoff_s=5.0)],
    config=make_pipeline_config(),
)
dets_3_inh = pipe_3_inh.run_offline()
sw_dets_3 = get_detections(dets_3_inh)
stims_3_inh = get_stims(dets_3_inh)

seqs_inh = defaultdict(list)
for e in stims_3_inh:
    seqs_inh[e.metadata['detection_time']].append(e)

complete = sum(1 for s in seqs_inh.values() if len(s) == 3)
incomplete = sum(1 for s in seqs_inh.values() if 0 < len(s) < 3)
no_stims = len(sw_dets_3) - len(seqs_inh)

print(f'3-pulse + IED inhibition:')
print(f'  {len(sw_dets_3)} detections')
print(f'  {complete} complete (all 3 stims)')
print(f'  {incomplete} partial (some cancelled by IED)')
print(f'  {no_stims} fully blocked')
print(f'  {len(stims_3_inh)} total stim events')

---
## 5. Detection Report

Three-panel summary:
(a) stim-triggered average — should show slow-wave morphology centred on the peak
(b) stim phase distribution — should cluster near 0 (peak)
(c) inhibition summary

In [ ]:
# Run full pipeline with inhibition
pipe_report = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(), make_inhibitor(),
             make_trigger(n_pulses=1, backoff_s=2.5)],
    config=make_pipeline_config(),
)
events_report = pipe_report.run_offline()
dets_report = get_detections(events_report)
stims_report = get_stims(events_report, pulse_index=1)

sr = CFG['pipeline']['sample_rate']
win_s = 2.0
win_samples = int(win_s * sr)
t_win = np.arange(-win_samples, win_samples) / sr

# (a) Stim-triggered waveforms (centred on STIM time = predicted peak)
stim_windows = []
for e in stims_report:
    centre = int(e.timestamp * sr)
    lo, hi = centre - win_samples, centre + win_samples
    if lo >= 0 and hi < len(sig_ied):
        stim_windows.append(sig_ied[lo:hi])

# (b) Detection phases (should be near π = trough)
det_phases = np.array([d.metadata.get('detection_phase', 0.0) for d in dets_report])

# (c) Inhibition comparison
cooldown = CFG['trigger']['inhibition_cooldown_s']
pipe_noinh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[make_wavelet(), make_detector(),
             make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=5.0)],
    config=make_pipeline_config(),
)
stims_noinh = get_stims(pipe_noinh.run_offline(), pulse_index=1)

blocked_stims = []
for s in stims_noinh:
    if not any(abs(s2.timestamp - s.timestamp) < 0.1 for s2 in stims_report):
        nearest_ied = min((abs(s.timestamp - ied.timestamp) for ied in ied_gt), default=999)
        blocked_stims.append((s.timestamp, nearest_ied))

correct_blocks = sum(1 for _, d in blocked_stims if d < cooldown)
wrong_blocks = sum(1 for _, d in blocked_stims if d >= cooldown)

print(f'Without inhibition: {len(stims_noinh)} stims')
print(f'With inhibition:    {len(stims_report)} stims')
print(f'Blocked: {len(blocked_stims)} ({correct_blocks} near IED, {wrong_blocks} no nearby IED)')

# ── Plot ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 5))

# (a) Stim-triggered average
ax_a = fig.add_subplot(1, 3, 1)
for w in stim_windows:
    ax_a.plot(t_win, w, alpha=0.3, lw=0.5)
if stim_windows:
    avg = np.mean(stim_windows, axis=0)
    ax_a.plot(t_win, avg, 'k-', lw=2, label='mean')
ax_a.axvline(0, color='red', ls='--', alpha=0.6, lw=1, label='stim (peak)')
ax_a.set_xlabel('Time from stim (s)')
ax_a.set_ylabel('Amplitude')
ax_a.set_title(f'Stim-triggered average (n={len(stim_windows)})')
ax_a.legend(fontsize=9)

# (b) Detection phase polar plot (should cluster at π = trough)
ax_b = fig.add_subplot(1, 3, 2, projection='polar')
if len(det_phases) > 0:
    n_bins = 24
    counts, edges = np.histogram(det_phases, bins=n_bins, range=(0, 2*pi))
    centres = (edges[:-1] + edges[1:]) / 2
    width = 2 * pi / n_bins
    ax_b.bar(centres, counts, width=width, alpha=0.7, edgecolor='k', lw=0.5)
    mean_phase = np.angle(np.mean(np.exp(1j * det_phases)))
    mean_r = np.abs(np.mean(np.exp(1j * det_phases)))
    ax_b.annotate('', xy=(mean_phase, max(counts) * 0.9), xytext=(mean_phase, 0),
                  arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax_b.set_title(f'Detection phase (n={len(det_phases)}, '
                   f'mean={mean_phase:.2f}, R={mean_r:.2f})\n'
                   f'Should cluster at π={pi:.2f} (trough)', pad=15)
else:
    ax_b.set_title('No detections')

# (c) Inhibition summary
ax_c = fig.add_subplot(1, 3, 3)
labels = ['Fired\n(no IED)', 'Blocked\n(IED nearby)', 'Blocked\n(no IED)']
values = [len(stims_report), correct_blocks, wrong_blocks]
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax_c.bar(labels, values, color=colors, edgecolor='k', lw=0.5)
for bar, val in zip(bars, values):
    if val > 0:
        ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                  str(val), ha='center', va='bottom', fontweight='bold')
ax_c.set_ylabel('Count')
ax_c.set_title(f'Inhibition (cooldown={cooldown}s)')
ax_c.set_ylim(0, max(values + [1]) * 1.3)

plt.tight_layout()
plt.show()

---
## 6. Timing precision check

For a 1 Hz SW detected at the trough, the stim should arrive ~500ms later at the peak.
In offline mode, timing is quantised to `chunk_duration` — in live mode, the
`StimScheduler` thread handles sub-chunk precision.

In [ ]:
# For each detection with a stim, check the detection→stim delay
delays_ms = []
for d in dets_report:
    det_t = d.timestamp
    my_stim = next(
        (s for s in stims_report
         if abs(s.metadata.get('detection_time', -1) - det_t) < 0.001),
        None
    )
    if my_stim:
        delays_ms.append((my_stim.timestamp - det_t) * 1000)

if delays_ms:
    delays_ms = np.array(delays_ms)
    freqs = np.array([d.metadata['frequency'] for d in dets_report if any(
        abs(s.metadata.get('detection_time', -1) - d.timestamp) < 0.001
        for s in stims_report
    )])
    expected_ms = 1000 / (2 * freqs)  # half period at detected freq

    print(f'Detection→Stim delays (n={len(delays_ms)}):')
    print(f'  Actual:   {np.mean(delays_ms):.0f} ± {np.std(delays_ms):.0f} ms')
    print(f'  Expected: {np.mean(expected_ms):.0f} ± {np.std(expected_ms):.0f} ms (half period)')
    print(f'  Error:    {np.mean(delays_ms - expected_ms):.0f} ± {np.std(delays_ms - expected_ms):.0f} ms')
    print(f'  (chunk quantisation = {CFG["pipeline"]["chunk_duration"]*1000:.0f} ms)')
else:
    print('No detection→stim pairs found')